<h1><center>Классификация текстов с использованием Word2Vec</h1></center>

<h2>Загрузка и преобработка данных</h2>

In [1]:
data_path = 'D:/MIREA/DPO/Learning/Datasets/NLP_DisasterTweets/'

In [2]:
import pandas as pd
import numpy as np
import spacy
from spacy.lang.en.stop_words import STOP_WORDS
import re
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score, GridSearchCV

In [3]:
nlp = spacy.load('en_core_web_sm')
pd.options.display.max_colwidth = 400

In [4]:
train = pd.read_csv(data_path + 'train.csv')
test = pd.read_csv(data_path + 'test.csv')

print('Train shape: {}'.format(train.shape))
print('Test shape: {}'.format(test.shape))
train.head()

Train shape: (7613, 5)
Test shape: (3263, 4)


,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake May ALLAH Forgive us all,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are being notified by officers. No other evacuation or shelter in place orders are expected,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation orders in California",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as smoke from #wildfires pours into a school,1


In [5]:
test.head()

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, stay safe everyone."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are fleeing across the street, I cannot save them all"
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan


In [6]:
nlp = spacy.load('en_core_web_lg', disable=['tagger', 'parcer', 'ner'])
docs_train = [nlp(doc).vector for doc in train.text]
X_train = np.vstack(docs_train)
print('Shape of train: {}'.format(X_train.shape))

docs_test = [nlp(doc).vector for doc in test.text]
X_test = np.vstack(docs_test)
print('Shape of test: {}'.format(X_test.shape))

y_train = train.target.copy()

d:\MIREA\DPO\.venv3_11_8\Lib\site-packages\spacy\pipeline\lemmatizer.py:187: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


Shape of train: (7613, 300)
Shape of test: (3263, 300)


In [7]:
X_train

array([[-0.27897874,  0.10392565, -0.09989621, ...,  0.04564636,
        -0.06325858, -0.05639894],
       [-0.02076824,  0.05306825, -0.15192825, ..., -0.05474687,
        -0.2549715 ,  0.11622399],
       [-0.01530766,  0.06607368, -0.09196319, ..., -0.0348906 ,
         0.07479551,  0.0337687 ],
       ...,
       [ 0.09417391,  0.07030873,  0.00669454, ...,  0.05192918,
        -0.00719345,  0.03631673],
       [-0.01930599,  0.18637651, -0.21350445, ..., -0.07282014,
         0.03912553,  0.07439488],
       [-0.08053643,  0.21419215, -0.02220372, ..., -0.24513498,
        -0.03821203,  0.16831529]], shape=(7613, 300), dtype=float32)

In [8]:
X_test

array([[-0.05050467,  0.30811986, -0.14809568, ..., -0.0081825 ,
         0.11909617,  0.06569584],
       [-0.02575657,  0.2921284 , -0.184004  , ...,  0.01653059,
        -0.11942641,  0.13354763],
       [ 0.01218249,  0.1292677 , -0.19094108, ..., -0.03635559,
        -0.01576087,  0.07680107],
       ...,
       [ 0.04517917,  0.16106233,  0.05652167, ...,  0.071525  ,
        -0.14023666,  0.2900405 ],
       [-0.04674022,  0.11613716, -0.04321733, ..., -0.18580699,
        -0.02813433, -0.13857909],
       [-0.01196804,  0.18542053,  0.12331672, ..., -0.06519318,
        -0.11748392,  0.00093855]], shape=(3263, 300), dtype=float32)

<h2>Пайплайн машинного обучения

In [9]:
word2vec_pipe = Pipeline([('estimator', LogisticRegression())])
print('F1 score: {:.3f}'.format(np.mean(cross_val_score(word2vec_pipe, X_train,
                                                        y_train, scoring='f1'))))

word2vec_pipe.fit(X_train, y_train)
pred = word2vec_pipe.predict(X_test)
pred

F1 score: 0.728


array([1, 1, 1, ..., 1, 1, 1], shape=(3263,))

In [10]:
np.unique(pred)

array([0, 1])

<h2>Тюнинг гиперпараметров

In [15]:
param_grid = [{'estimator': [LogisticRegression()],
               'estimator__C': np.logspace(-5, 5, 10)},
               {'estimator': [SVC()],
                'estimator__C': np.logspace(-3, 3, 5),
                'estimator__gamma': np.logspace(-3, 3, 5) / X_train.shape[0]}]

word2vec_grid_search = GridSearchCV(
    estimator=word2vec_pipe,
    param_grid=param_grid,
    scoring='f1',
    n_jobs=-1,
    refit=True,
    verbose=1,
    return_train_score=True
)

word2vec_grid_search.fit(X_train, y_train)

cols = ['param_estimator',
        'param_estimator__C',
        'param_estimator__gamma',
        'mean_test_score',
        'mean_train_score']

pd.options.display.max_colwidth = 50

word2vec_grid_search_results = pd.DataFrame(word2vec_grid_search.cv_results_).sort_values(by='mean_test_score', ascending=False)
word2vec_grid_search_results[cols]

Fitting 5 folds for each of 35 candidates, totalling 175 fits


,param_estimator,param_estimator__C,param_estimator__gamma,mean_test_score,mean_train_score
24,SVC(),1.000000,1.313543e-01,0.736156,0.812053
28,SVC(),31.622777,4.153786e-03,0.732952,0.788173
4,LogisticRegression(),0.278256,NaN,0.732219,0.775045
33,SVC(),1000.000000,4.153786e-03,0.729643,0.832892
32,SVC(),1000.000000,1.313543e-04,0.724860,0.781138
5,LogisticRegression(),3.593814,NaN,0.724451,0.792961
7,LogisticRegression(),599.484250,NaN,0.717789,0.793701
8,LogisticRegression(),7742.636827,NaN,0.717737,0.793681
9,LogisticRegression(),100000.000000,NaN,0.716619,0.794104
6,LogisticRegression(),46.415888,NaN,0.716343,0.793370
